# V18 VTC PCB test: AWG voltage in, FPGA pulse width out

Test the voltage-to-time converter **without the matrix**. Run cells in order.
Only the acquisition cell opens instruments/enables outputs. Edit the voltage
limits and stimulus settings to suit the PCB first.

| Connection | Purpose |
|---|---|
| Rigol CH1 → PCB voltage input | DC, sine, square, ramp, triangle or uploaded arbitrary waveform |
| FPGA DIO1_P / E1 pin 5 → PCB trigger | Fixed **100 kHz**, 50% duty, generated by FPGA PULSE |
| PCB pulse output → FPGA DIO2_P / E1 pin 7 | SINGLE TDC input, 0–3.3 V logic |
| Common signal ground | AWG, PCB and FPGA reference |

Disconnect the old direct AWG-to-DIO2_P cable. Both FPGA analogue outputs stay
OFF; the matrix and its two weights are absent. The PCB must produce one positive
return pulse per FPGA launch, with both edges inside its 10 µs TDC frame.

The AWG controls **voltage**, not the trigger/frame clock. Its initial waveform
phase need not match the FPGA. That changes which voltage is sampled, not the
causal launch/return association. No independent-source phase search, startup
retry or AWG-derived width-bias correction is used. Voltage-dependent PCB delay
and width must still fit the frame across the entire chosen input range.

Uses the qualified exact-route V18 SINGLE fine LUTs bundled in `100kHz_demo`.
Requires NumPy, Matplotlib, PyVISA and your USB VISA runtime. Keep the matching
V18 bitstream/API/server loaded. Software cannot verify the routed-image hash
from the version register alone.


In [ ]:
from pathlib import Path
import sys, json, time, hashlib
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt

REPO = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p/'scpi-tests'/'vtc_pcb_tools.py').is_file()), None)
if REPO is None: raise RuntimeError('Start Jupyter in the repository or scpi-tests folder')
sys.path.insert(0, str(REPO/'scpi-tests'))
from tdc_rigol_tools import Rigol
from vtc_pcb_tools import load_demo_helpers, prepare_stimulus, configure_stimulus, stop_vtc, capture_vtc
demo = load_demo_helpers(REPO)  # imports/definitions only; no connection

RP_HOST = '169.254.77.151'
RP_PORT = 5000
BITSTREAM_LABEL = 'v18_20261026_140711'
AWG_CONFIG = dict(visa_resource='USB0::0x1AB1::0x0647::DG5R270700006::INSTR',
                  visa_backend='', awg_channel=1, awg_serial='DG5R270700006',
                  visa_timeout_ms=10000, awg_command_delay_s=0.02)
FRAME_PERIOD_NS = 10000        # fixed: do not match this to AWG waveform frequency
FRAME_OFFSET_NS = 0           # PCB frame boundary, configured once; 4 ns grid
FRAMES = 8192                 # 81.92 ms; maximum 16384 (163.84 ms)
SETTLE_S = 0.10               # adjust for PCB voltage settling
PCB_MIN_V, PCB_MAX_V = 0.0, 1.0  # EDIT to the actual PCB input operating range
BUNDLE = REPO/'100kHz_demo'
RESULTS_DIR = demo.DATA_ROOT/'scpi-tests'/'vtc_pcb_results'

# Choose the test waveform. Vpp is peak-to-peak at High-Z; offset_v is DC bias.
STIMULUS = dict(kind='SINE', frequency_hz=1000, vpp=0.2, offset_v=0.5)
# STIMULUS = dict(kind='SQUARE', frequency_hz=100, vpp=0.4, offset_v=0.5)  # 50% duty
# STIMULUS = dict(kind='DC', offset_v=0.5)
# STIMULUS = dict(kind='RAMP', frequency_hz=100, vpp=0.4, offset_v=0.5)    # 50% symmetry
# STIMULUS = dict(kind='TRIANGLE', frequency_hz=100, vpp=0.4, offset_v=0.5)

# A custom PERIODIC waveform: normalized samples, not physical volts.
# For measured data: samples = np.loadtxt('my_wave.csv', delimiter=',')
phase = np.arange(1024)/1024
custom_samples = 0.7*np.sin(2*np.pi*phase) + 0.2*np.sin(6*np.pi*phase)
# STIMULUS = dict(kind='ARB', frequency_hz=100, vpp=0.4,
#                 offset_v=0.5, samples=custom_samples.tolist())

# Optional static transfer measurement after the chosen waveform capture.
RUN_DC_SWEEP = False
DC_LEVELS_V = np.linspace(0.3, 0.7, 9).tolist()


## Check the plan and load calibration on the PC

The 100 kHz trigger stays fixed for every waveform and DC point. For initial
dynamic tests use a slow waveform (e.g. 100 Hz–1 kHz); the TDC samples one width
per 10 µs. Square/ARB edges can cross a sampling instant, and harmonics can alias.
The helper limits fundamental frequency below 50 kHz; this does not guarantee
band-limited input. AWG frequency and phase are never used to change FPGA timing.

For ARB, supply 32–16384 samples within [-1,1]. They are periodically interpolated
to 16384 points for Rigol basic-mode playback, without automatic normalization.
`offset_v + vpp/2 * sample` sets nominal voltage. Upload overwrites only
`INT:\vtc_pcb_test.arb` on the AWG; the actual sample list is saved in the test plan.
No AWG reset is sent. Only the selected AWG channel is controlled.
The High-Z voltage convention assumes a high-impedance PCB input; account for
termination when setting limits.

The LUT calibrates FPGA fine timing, not the PCB voltage-to-width transfer.
The optional DC sweep measures that transfer against nominal AWG voltage; an
independent voltage measurement is needed to establish absolute voltage accuracy.


In [ ]:
manifest = json.loads((BUNDLE/'v18_calibration_manifest.json').read_text())
if manifest['bitstream_label'] != BITSTREAM_LABEL:
    raise ValueError('Calibration belongs to another routed build')
for name, digest in manifest['files_sha256'].items():
    if hashlib.sha256((BUNDLE/name).read_bytes()).hexdigest() != digest:
        raise ValueError('Calibration bundle checksum mismatch: '+name)
calibration = demo.read_calibration(BUNDLE/'v18_luts.json')
cases = [dict(label='waveform', stimulus=STIMULUS)]
if RUN_DC_SWEEP:
    cases += [dict(label=f'dc_{i:02d}', stimulus=dict(kind='DC', offset_v=v))
              for i,v in enumerate(DC_LEVELS_V)]
for case in cases:
    info, _ = prepare_stimulus(case['stimulus'], PCB_MIN_V, PCB_MAX_V)
    print(case['label'], {k:v for k,v in info.items() if k != 'samples_sha256'})
assert FRAME_PERIOD_NS == 10000
print(f'FPGA launch: 100 kHz; {FRAMES} frames = {FRAMES/100:.2f} ms per burst')


## Optional fixed return-window check

When PCB timing is known, uncomment the call and supply measured bounds in ns
relative to the TDC launch/frame reference, including path delay and jitter.
Check the full voltage range. A common fixed delay cancels in width; neither
edge may reach a frame boundary. Do not substitute historical AWG frame offsets.


In [ ]:
# demo.check_pcb_return_window(earliest_rise_ns, latest_rise_ns, maximum_width_ns,
#     period_ns=FRAME_PERIOD_NS, offset_ns=FRAME_OFFSET_NS, guard_ns=8)


## Acquire: setup once, upload both LUTs, then run the chosen cases

This cell connects to hardware. The visible sequence keeps the FPGA launch
running at 100 kHz between cases, changes only the AWG voltage waveform, waits
for settling, then captures **once**. LUTs and offset are reused throughout.
On the first incomplete burst it stops the sequence; it does not phase-retry.
Both analogue FPGA outputs remain OFF. Each case gets its own settings and
status summary. Only successful fully complete bursts retain raw CSV files;
unsuccessful attempts retain compact JSON diagnostics. Setup/configuration
exceptions go into `session.json`, even if capture has not started.

The `finally` block tries to stop FPGA launch, both FPGA analogue outputs and
the selected AWG output, then closes both sessions. To acquire another batch,
edit settings and rerun plan + acquisition; LUT upload here is a batch setup
step, never a per-capture step. After a connection loss, check output states
before continuing; the final recovery cell can issue OFF on fresh sessions.


In [ ]:
RUN_DIR = RESULTS_DIR/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
plan = dict(bitstream_label=BITSTREAM_LABEL, topology='fpga_triggered_vtc_pcb',
            frame_period_ns=10000, frame_offset_ns=FRAME_OFFSET_NS, frames=FRAMES,
            pcb_voltage_limits_v=[PCB_MIN_V, PCB_MAX_V], settle_s=SETTLE_S,
            awg_config=AWG_CONFIG, cases=cases, automatic_retries=0)
(RUN_DIR/'plan.json').write_text(json.dumps(plan, indent=2))
rp = awg = None
captures = []
session_error = None
cleanup_errors = []
try:
    rp = demo.SCPI(RP_HOST, RP_PORT)
    image = demo.prepare_tdc(rp)            # parks FPGA analogue outputs OFF
    demo.configure_launch(rp, 10000)       # fixed period=10000, HIGH=5000 ns
    demo.set_frame_offset(rp, FRAME_OFFSET_NS)
    demo.upload_luts(rp, calibration)      # both tables + readback, once
    demo.enable_fine_mode(rp)
    awg = Rigol(AWG_CONFIG)
    awg.off()
    demo.start_launch(rp)
    for case in cases:
        settings = configure_stimulus(awg, case['stimulus'], PCB_MIN_V, PCB_MAX_V)
        awg.on()
        time.sleep(SETTLE_S)
        capture = capture_vtc(demo, rp, FRAMES, calibration)
        capture.update(label=case['label'], stimulus=settings, image=image,
                       bitstream_label=BITSTREAM_LABEL, matrix_present=False)
        captures.append(capture)
        print(case['label'], capture['summary'])
        if not capture['success']:
            raise RuntimeError('Incomplete/invalid VTC burst; inspect flags, pulse presence and fixed frame placement')
except BaseException as exc:
    session_error = f'{type(exc).__name__}: {exc}'
    raise
finally:
    cleanup_errors = stop_vtc(demo, rp, awg)
    # Close both sessions even if one close raises.
    for name, instrument in (('AWG', awg), ('FPGA', rp)):
        if instrument is not None:
            try:
                instrument.close() if name == 'AWG' else instrument.__exit__(None, None, None)
            except Exception as exc:
                cleanup_errors.append(f'{name} close: {exc}')
    for capture in captures:
        capture['cleanup_errors'] = list(cleanup_errors)
        if cleanup_errors: capture['success'] = False
        capture['capture_files'] = demo.save_capture(capture, RUN_DIR)
    session = dict(error=session_error, cleanup_errors=cleanup_errors,
                   attempts=len(captures), awg_identity=getattr(awg,'identity',None),
                   awg_startup_errors=getattr(awg,'startup_errors',[]))
    (RUN_DIR/'session.json').write_text(json.dumps(session, indent=2))
    print('Saved:', RUN_DIR)
    if cleanup_errors: print('Cleanup needs attention:', cleanup_errors)


## Inspect pulse widths and the optional static transfer curve

Plots preserve invalid frames rather than selecting favourable records. Width
variation during a dynamic waveform is the expected PCB signal, not a TDC
precision estimate. Since the AWG voltage waveform is not phase-locked or sampled
by an ADC here, do not label individual widths with a calculated instantaneous
AWG voltage. The DC curve can pair nominal voltage with mean width directly.
Square-wave transitions may show mixed samples or PCB settling.


In [ ]:
for capture in captures:
    print(capture['label'], 'success:', capture['success'], capture['errors'])
    demo.plot_capture(capture)

dc = [c for c in captures if c['success'] and c['stimulus']['kind']=='DC']
if dc:
    voltages = [c['stimulus']['offset_v'] for c in dc]
    widths = [np.array([r['width_ps']/1000 for r in c['records']]) for c in dc]
    fig, ax = plt.subplots(figsize=(7,4))
    ax.errorbar(voltages, [w.mean() for w in widths],
                yerr=[w.std() for w in widths], fmt='o-', capsize=3)
    ax.set(xlabel='Nominal AWG input (V, High-Z)', ylabel='PCB pulse width (ns)',
           title='VTC static transfer; error bars = burst SD')
    ax.grid(True); fig.tight_layout()
    fig.savefig(RUN_DIR/'dc_transfer.png', dpi=160)


## Recovery after a stopped cell / connection loss (run only when needed)

Reconnect to request OFF; this does not clear LUTs or the frame offset.
After FPGA reload/power loss rerun the acquisition setup. This cell is commented
so Run All does not open a second session unnecessarily.

Rigol SCPI implementation follows the manufacturer's
[DG5000 Pro programming guide](https://www.batterfly.com/PDF/RIGOL/DG5000-Pro/DG5000-Pro-Series_programmingguide_EN.pdf),
sections 3.9.8, 3.9.16 and 3.12.2. Basic-mode stored ARB uses 16384 points.


In [ ]:
# print(demo.stop_board(RP_HOST, RP_PORT))
# recovery_awg = Rigol(AWG_CONFIG)
# try:
#     recovery_awg.off()
# finally:
#     recovery_awg.close()
